# Disaster Recovery Cost Prediction and Resilience Optimization
## Week 2 Day 5 — XGBoost Tuning and Week 2 Checkpoint

## Goal of this notebook

The purpose of this notebook is to tune the XGBoost model using GridSearchCV, compare the tuned model against the existing Random Forest and Linear Regression baselines on a held-out test set, and decide whether the tuned XGBoost model should replace the current best model.

This notebook also serves as a Week 2 checkpoint by confirming that:
- the processed dataset is ready
- three models have been trained
- evaluation and SHAP analysis have been completed
- the modelling workflow is reproducible and ready for Week 3 deployment work

In [1]:
import json
import pickle
from pathlib import Path

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from xgboost import XGBRegressor

In [2]:
PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "processed_disasters.csv"
MODEL_PATH = PROJECT_ROOT / "models" / "best_model.pkl"
METADATA_PATH = PROJECT_ROOT / "models" / "best_model_metadata.json"

MLFLOW_TRACKING_URI = (PROJECT_ROOT / "mlruns").resolve().as_uri()
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_registry_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment("disaster_recovery_cost_prediction")

C:\Users\caspe\anaconda\envs\genexa_ds\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:177: FutureWarning: The filesystem tracking backend (e.g., './mlruns') will be deprecated in February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://github.com/mlflow/mlflow/issues/18534 for more details and migration guidance.
  return FileStore(store_uri, store_uri)


<Experiment: artifact_location='file:///C:/Users/caspe/OneDrive/Documents/disaster-recovery-cost-prediction/mlruns/664847857236199730', creation_time=1775714010057, experiment_id='664847857236199730', last_update_time=1775714010057, lifecycle_stage='active', name='disaster_recovery_cost_prediction', tags={}>

## Section B — Load the Processed Dataset

This dataset contains one row per disaster and the cleaned modelling features developed during Week 2.

In [3]:
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (5163, 43)


,disasterNumber,state,incidentType,declarationType,declarationDate,incidentBeginDate,incidentEndDate,declaration_year,declaration_month,incident_duration_days,...,designatedArea,lastIAFilingDate,designatedIncidentTypes,lastRefresh,hash,id,totalObligatedAmountPa,totalObligatedAmountCatAb,totalObligatedAmountCatC2g,totalObligatedAmountHmgp
0,1,GA,Tornado,DR,1953-05-02 00:00:00+00:00,1953-05-02 00:00:00+00:00,1953-05-02 00:00:00+00:00,1953,5,0.0,...,Statewide,NaN,NaN,2024-08-27T18:22:14.800Z,413ff808d79f08a6710f6b78f361d5a7de692711,8943dfcf-9786-4e51-8889-d62014034bb2,NaN,NaN,NaN,NaN
1,2,TX,Tornado,DR,1953-05-15 00:00:00+00:00,1953-05-15 00:00:00+00:00,1953-05-15 00:00:00+00:00,1953,5,0.0,...,Statewide,NaN,W,2024-08-27T18:22:14.800Z,8a8bc885c003cb873c201bb6a3a2771a6d84efb1,ff821327-6b90-4246-b19f-fff8c4b288a8,NaN,NaN,NaN,NaN
2,3,LA,Flood,DR,1953-05-29 00:00:00+00:00,1953-05-29 00:00:00+00:00,1953-05-29 00:00:00+00:00,1953,5,0.0,...,Statewide,NaN,NaN,2024-08-27T18:22:14.800Z,b6e6f19ae3c0d2383b7b873b8495bd2770f2ff9a,cd461e08-5ac9-4e70-8507-9c7a3cbff265,NaN,NaN,NaN,NaN
3,4,MI,Tornado,DR,1953-06-02 00:00:00+00:00,1953-06-02 00:00:00+00:00,1953-06-02 00:00:00+00:00,1953,6,0.0,...,Statewide,NaN,NaN,2024-08-27T18:22:14.800Z,34f0061012c8069f145d56a3537cd327b7d4e49b,53be0c04-d2ae-42fb-b070-a01b0a50b7f6,NaN,NaN,NaN,NaN
4,5,MT,Flood,DR,1953-06-06 00:00:00+00:00,1953-06-06 00:00:00+00:00,1953-06-06 00:00:00+00:00,1953,6,0.0,...,Statewide,NaN,NaN,2024-08-27T18:22:14.800Z,3bdbec258e4640c3f02971dbc1f9dbc3ebbfc96a,4b3ed0ac-299b-49f0-80d4-9a2a6bacd5a4,NaN,NaN,NaN,NaN


## Section C — Define Target and Features

The target remains `log_total_obligated_amount`, and the same leakage-safe exclusions from Day 3 are used here.

In [4]:
target_col = "log_total_obligated_amount"

exclude_cols = {
    # target / leakage
    "log_total_obligated_amount",
    "total_obligated_amount",
    "totalObligatedAmountPa",
    "totalObligatedAmountCatAb",
    "totalObligatedAmountCatC2g",
    "totalObligatedAmountHmgp",
    "project_count",
    "avg_project_amount",

    # identifiers
    "disasterNumber",
    "id",
    "hash",
    "femaDeclarationString",
    "declarationRequestNumber",
    "incidentId",
    "lastRefresh",

    # post-decision program flags
    "ihProgramDeclared",
    "iaProgramDeclared",
    "paProgramDeclared",
    "hmProgramDeclared",

    # raw dates
    "declarationDate",
    "incidentBeginDate",
    "incidentEndDate",
    "disasterCloseoutDate",
    "lastIAFilingDate",

    # high-cardinality text
    "declarationTitle",
    "designatedArea",
    "designatedIncidentTypes",
}

candidate_features = [col for col in df.columns if col not in exclude_cols]

numeric_features = []
categorical_features = []

for col in candidate_features:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_features.append(col)
    else:
        categorical_features.append(col)

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)

Numeric features: ['declaration_year', 'declaration_month', 'incident_duration_days', 'state_5yr_disaster_count', 'high_cost_incident', 'fyDeclared', 'tribalRequest', 'fipsStateCode', 'fipsCountyCode', 'placeCode', 'region']
Categorical features: ['state', 'incidentType', 'declarationType', 'season', 'census_region']


In [5]:
X = df[numeric_features + categorical_features].copy()
y = df[target_col].copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

Train shape: (4130, 16)
Test shape: (1033, 16)


## Section D — Build Preprocessing Pipeline

Numeric features are median-imputed and standardized, while categorical features are mode-imputed and one-hot encoded.

In [6]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

## Section E — Build Baseline Pipelines

Three models are compared:
- Linear Regression
- Random Forest
- Tuned XGBoost

In [7]:
linear_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression()),
    ]
)

rf_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", RandomForestRegressor(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        )),
    ]
)

xgb_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", XGBRegressor(
            random_state=42,
            n_jobs=-1,
            objective="reg:squarederror",
            eval_metric="rmse"
        )),
    ]
)

## Section F — XGBoost Tuning

GridSearchCV is used to search across a small but meaningful set of XGBoost hyperparameters.

In [8]:
param_grid = {
    "model__n_estimators": [200, 300],
    "model__max_depth": [4, 6, 8],
    "model__learning_rate": [0.01, 0.05, 0.1],
}

cv = KFold(n_splits=5, shuffle=True, random_state=42)

xgb_grid = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=param_grid,
    scoring="r2",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    refit=True,
)

xgb_grid.fit(X_train, y_train)

print("Best XGBoost params:", xgb_grid.best_params_)
print("Best XGBoost CV R2:", xgb_grid.best_score_)

Fitting 5 folds for each of 18 candidates, totalling 90 fits
Best XGBoost params: {'model__learning_rate': 0.05, 'model__max_depth': 4, 'model__n_estimators': 200}
Best XGBoost CV R2: 0.8469000265466251


## Section G — Log Grid Search Results to MLflow

The tuning run, best parameters, and best score are logged to MLflow for reproducibility.

In [9]:
with mlflow.start_run(run_name="xgboost_gridsearch_tuning"):
    mlflow.log_param("model_name", "xgboost_tuned")
    mlflow.log_param("param_grid", str(param_grid))
    mlflow.log_param("n_numeric_features", len(numeric_features))
    mlflow.log_param("n_categorical_features", len(categorical_features))

    for param_name, param_value in xgb_grid.best_params_.items():
        mlflow.log_param(param_name, param_value)

    mlflow.log_metric("best_cv_r2", float(xgb_grid.best_score_))

# ⚙️ Section G — XGBoost Grid Search Findings

The XGBoost hyperparameter tuning was performed using GridSearchCV across a controlled parameter grid:

- `n_estimators`: [200, 300]  
- `max_depth`: [4, 6, 8]  
- `learning_rate`: [0.01, 0.05, 0.1]  

The best-performing configuration was:

- **n_estimators = 200**  
- **max_depth = 4**  
- **learning_rate = 0.05**

This combination achieved a cross-validated R² score of approximately **0.8469**, indicating strong predictive performance while maintaining generalisation.

Notably, the optimal configuration favours:
- a **shallower tree depth**, suggesting that overly complex trees do not generalise well  
- a **moderate learning rate**, balancing convergence speed and stability  
- a **moderate number of estimators**, avoiding unnecessary model complexity  

**Key insights:**
- Simpler XGBoost configurations generalise better than deeper trees  
- Model performance stabilises around ~0.84–0.85 R² after leakage removal  
- Tuning provides marginal but meaningful improvements over default settings  
- Cross-validation results are now realistic and no longer inflated  

## Section H — Held-out Test Set Comparison

The tuned XGBoost model is compared against Random Forest and Linear Regression on the same unseen test set.

In [10]:
linear_pipeline.fit(X_train, y_train)
rf_pipeline.fit(X_train, y_train)
tuned_xgb_pipeline = xgb_grid.best_estimator_

In [11]:
def evaluate_model(name, model, X_test, y_test):
    y_pred = model.predict(X_test)
    return {
        "model_name": name,
        "test_r2": r2_score(y_test, y_pred),
        "test_rmse": np.sqrt(mean_squared_error(y_test, y_pred)),
        "test_mae": mean_absolute_error(y_test, y_pred),
    }

comparison_results = pd.DataFrame([
    evaluate_model("linear_regression", linear_pipeline, X_test, y_test),
    evaluate_model("random_forest", rf_pipeline, X_test, y_test),
    evaluate_model("xgboost_tuned", tuned_xgb_pipeline, X_test, y_test),
]).sort_values("test_r2", ascending=False).reset_index(drop=True)

display(comparison_results)

,model_name,test_r2,test_rmse,test_mae
0,random_forest,0.893947,2.919370,1.199830
1,xgboost_tuned,0.890772,2.962738,1.365014
2,linear_regression,0.639206,5.384634,4.051451


# 🧪 Section H — Held-out Test Set Comparison Findings

The tuned XGBoost model was compared against Random Forest and Linear Regression using a held-out test set.

### Test Set Performance Summary:

| Model              | R²     | RMSE   | MAE    |
|-------------------|--------|--------|--------|
| Random Forest     | 0.8940 | 2.9194 | 1.1989 |
| XGBoost (tuned)   | 0.8908 | 2.9674 | 1.3650 |
| Linear Regression | 0.6392 | 5.3846 | 4.0515 |

### Interpretation

- **Random Forest achieved the highest R² and lowest MAE**, making it the best-performing model on unseen data.
- **Tuned XGBoost performed closely**, but slightly underperformed Random Forest across all metrics.
- **Linear Regression performed significantly worse**, confirming that the problem is highly non-linear.

Although XGBoost performed well during cross-validation, it did not surpass Random Forest on the held-out test set, indicating that:

- Random Forest generalises slightly better for this dataset  
- XGBoost may require a larger or more refined hyperparameter search space for further gains  

**Key insights:**
- Random Forest remains the best overall model  
- XGBoost is competitive but not superior after tuning  
- Linear Regression serves as a weak baseline  
- Model performance is now realistic and consistent across CV and test data  

## Section I — Update Best Model if Needed

If tuned XGBoost outperforms the existing alternatives on held-out test R², it replaces the current best model artifact.

In [12]:
winner = comparison_results.iloc[0]["model_name"]
print("Winning model on test set:", winner)

Winning model on test set: random_forest


In [13]:
if winner == "xgboost_tuned":
    with open(MODEL_PATH, "wb") as f:
        pickle.dump(tuned_xgb_pipeline, f)

    updated_metadata = {
        "best_model_name": "xgboost_tuned",
        "target_column": target_col,
        "numeric_features": numeric_features,
        "categorical_features": categorical_features,
        "best_params": xgb_grid.best_params_,
        "best_cv_r2": float(xgb_grid.best_score_),
        "test_results": comparison_results.to_dict(orient="records"),
    }

    with open(METADATA_PATH, "w", encoding="utf-8") as f:
        json.dump(updated_metadata, f, indent=2)

    print("Updated best_model.pkl with tuned XGBoost.")
else:
    print("Tuned XGBoost did not win. Existing best model remains unchanged.")

Tuned XGBoost did not win. Existing best model remains unchanged.


# 🧾 Final Model Selection Decision

Based on the held-out test set evaluation:

- **Random Forest is retained as the best model**
- Tuned XGBoost does **not replace** the current `best_model.pkl`

This decision ensures that the deployed model reflects the strongest generalisation performance rather than just cross-validation results.

---

# 📊 Overall Tuning Conclusion

The tuning process successfully improved model robustness and validated that:

- previous extremely high scores (~0.99+) were due to leakage and have now been corrected  
- current performance (~0.89 R²) reflects a **realistic and reliable model**  
- Random Forest is well-suited to capturing complex, non-linear disaster cost relationships  
- XGBoost remains a strong alternative but requires further tuning to outperform Random Forest  

This completes the transition from:
- **inflated, misleading models → validated, production-ready models**


## Section J — Verify Saved Artifacts

This section checks that the model artifact and metadata file exist after tuning.

In [14]:
print("best_model.pkl exists:", MODEL_PATH.exists())
print("best_model_metadata.json exists:", METADATA_PATH.exists())

with open(MODEL_PATH, "rb") as f:
    loaded_model = pickle.load(f)

print("Loaded saved model successfully:", loaded_model is not None)

best_model.pkl exists: True
best_model_metadata.json exists: True
Loaded saved model successfully: True


# 🧪 Section J — Model Artifact Verification Findings

The final step of the pipeline involved verifying that the trained model and its associated metadata were successfully saved and can be reloaded without errors.

The following checks were performed:

- Confirmed that `best_model.pkl` exists → ✅ True  
- Confirmed that `best_model_metadata.json` exists → ✅ True  
- Successfully loaded the saved model using `pickle` → ✅ True  

### Interpretation

- The trained model pipeline (including preprocessing and estimator) has been **successfully persisted to disk**  
- The metadata file ensures **traceability**, capturing model configuration, features, and evaluation results  
- Successful deserialization confirms that the model is **ready for reuse in inference or deployment environments**

This step validates that the training workflow produces **reproducible and deployable artifacts**, which is critical for real-world machine learning systems.

## Section K — Week 2 Checkpoint

This section confirms that the key deliverables for Week 2 are complete.

In [15]:
checkpoint_items = pd.DataFrame({
    "item": [
        "Processed disaster-level dataset created",
        "Three models trained",
        "Leakage issues identified and fixed",
        "Model evaluation completed",
        "SHAP analysis completed",
        "XGBoost tuning completed",
        "Best model artifact saved",
    ],
    "status": [
        True,
        True,
        True,
        True,
        True,
        True,
        MODEL_PATH.exists(),
    ]
})

display(checkpoint_items)

,item,status
0,Processed disaster-level dataset created,True
1,Three models trained,True
2,Leakage issues identified and fixed,True
3,Model evaluation completed,True
4,SHAP analysis completed,True
5,XGBoost tuning completed,True
6,Best model artifact saved,True


# 🚀 Section K — Week 2 Checkpoint Summary

This section confirms that all key deliverables for Week 2 of the project have been successfully completed.

### Completed Milestones

- Processed disaster-level dataset created → ✅  
- Three models trained (Linear Regression, Random Forest, XGBoost) → ✅  
- Data leakage issues identified and resolved → ✅  
- Model evaluation completed (CV + test set) → ✅  
- SHAP explainability analysis implemented → ✅  
- XGBoost hyperparameter tuning completed → ✅  
- Final model artifact saved (`best_model.pkl`) → ✅  

### Interpretation

All critical components of the machine learning pipeline are now in place:

- A **clean, feature-engineered dataset**
- A **validated and leakage-free modelling approach**
- A **robust evaluation framework**
- A **selected best-performing model**
- **Explainability outputs (SHAP)**
- **Saved artifacts ready for deployment**

This marks the completion of a full **end-to-end modelling workflow**, transitioning the project from data preparation and experimentation into a **production-ready state**.

# 🧾 Final Week 2 Conclusion

By the end of Week 2, the project has evolved into a complete and reproducible machine learning pipeline that includes:

- Data ingestion and validation  
- Feature engineering at the correct analytical level  
- Leakage-free model training and evaluation  
- Model comparison and selection  
- Hyperparameter tuning  
- Model explainability (SHAP)  
- Artifact persistence and verification  

The system is now fully prepared for the next stage, which may include:

- Deployment (API or dashboard)  
- Business interpretation and reporting  
- Scenario simulation and decision support  

This represents a strong, industry-aligned foundation for disaster recovery cost prediction and resilience optimisation.